In [1]:
# ============================================
# GOLD LAYER - Star Schema
# ============================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

spark = SparkSession.builder \
    .appName("Ecommerce-Gold-Layer") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark Session Started Successfully")

Spark Session Started Successfully


In [2]:
GOLD_PATH = "gold/"

os.makedirs(GOLD_PATH, exist_ok=True)

In [3]:
orders         = spark.read.parquet("silver/orders")
order_items    = spark.read.parquet("silver/order_items")
customers      = spark.read.parquet("silver/customers")
products       = spark.read.parquet("silver/products")
sellers        = spark.read.parquet("silver/sellers")

print("Silver tables loaded successfully")

Silver tables loaded successfully


In [4]:
print("Orders       :", orders.count())
print("Order Items  :", order_items.count())
print("Customers    :", customers.count())
print("Products     :", products.count())
print("Sellers      :", sellers.count())

Orders       : 99441
Order Items  : 112650
Customers    : 99441
Products     : 32951
Sellers      : 3095


In [5]:
fact_order_items = (
    order_items
    .join(
        orders,
        on="order_id",
        how="left"
    )
)

In [6]:
print("Rows after join :", fact_order_items.count())

Rows after join : 112650


In [16]:
fact_order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- is_anomaly: boolean (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- is_late: boolean (nullable = true)



In [10]:
fact_order_items = fact_order_items.withColumn(
    "delivery_days",F.datediff(F.col("order_delivered_customer_date") , F.col("order_purchase_timestamp"))
)

In [15]:
fact_order_items = fact_order_items.withColumn(
    "is_late",
   F.when(
    F.col("order_delivered_customer_date").isNull(),
    None
)
.when(
    F.col("order_delivered_customer_date") >
    F.col("order_estimated_delivery_date"),
    True
).otherwise(False)
)

In [17]:
fact_order_items = fact_order_items.select(
    "order_id",
    "order_item_id",
    "customer_id",
    "product_id",
    "seller_id",
    "order_purchase_timestamp",
    "order_status",
    "price",
    "freight_value",
    "delivery_days",
    "is_late",
    "is_anomaly"
)

In [18]:
dim_customers = customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"  
)
dim_customers.write.mode("overwrite").parquet("gold/dim_customers")

In [20]:
dim_products = products.select(
    "product_id",
    "product_category_name_en",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
    
)
dim_products.write.mode("overwrite").parquet("gold/dim_products")

In [22]:
dim_sellers = sellers.select(
    "seller_id",
    "seller_city",
    "seller_state"
)
dim_sellers.write.mode("overwrite").parquet("gold/dim_products")